In [1]:
# Import relevant libraries.
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.corpus import words
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load dataset. Change directory as required.
df = pd.read_csv('uk_speeches.csv')

In [3]:
df.head()

,reference,country,date,title,author,is_gov,text
0,r980915a_BOE,united kingdom,15/09/1998,Speech,george,0,"Thank you, Chairman. I'm actually very pleased..."
1,r981021b_BOE,united kingdom,21/10/1998,Britain in Europe,george,0,It's a great pleasure to be here in the beauti...
2,r981021a_BOE,united kingdom,21/10/1998,Impact of the recent turbulence in internation...,king,1,Few industries have suffered more from volatil...
3,r981101a_BOE,united kingdom,01/11/1998,"Economic policy, with and without forecasts",budd,0,My topic this evening is the use of forecasts ...
4,r981101b_BOE,united kingdom,01/11/1998,Inflation targeting in practice: the UK experi...,vickers,0,"Six years ago this week, sterling left the exc..."


In [4]:
df.country.value_counts()

united kingdom    1190
Name: country, dtype: int64

In [5]:
# Add a column to calculate the string length per speech.
df['len'] = df['text'].str.len()
df

,reference,country,date,title,author,is_gov,text,len
0,r980915a_BOE,united kingdom,15/09/1998,Speech,george,0,"Thank you, Chairman. I'm actually very pleased...",13731
1,r981021b_BOE,united kingdom,21/10/1998,Britain in Europe,george,0,It's a great pleasure to be here in the beauti...,24263
2,r981021a_BOE,united kingdom,21/10/1998,Impact of the recent turbulence in internation...,king,1,Few industries have suffered more from volatil...,13678
3,r981101a_BOE,united kingdom,01/11/1998,"Economic policy, with and without forecasts",budd,0,My topic this evening is the use of forecasts ...,27679
4,r981101b_BOE,united kingdom,01/11/1998,Inflation targeting in practice: the UK experi...,vickers,0,"Six years ago this week, sterling left the exc...",27693
...,...,...,...,...,...,...,...,...
1185,r221007a_BOE,united kingdom,07/10/2022,"Shocks, inflation, and the policy response",ramsden,0,Thank you very much for the invitation to spea...,24773
1186,r221012a_BOE,united kingdom,12/10/2022,Monetary policy: an anchor in challenging times,pill,0,Huw Pill talks about how we will bring inflati...,22398
1187,r221015a_BOE,united kingdom,15/10/2022,Monetary policy and financial stability interv...,bailey,1,We are meeting in the most difficult of times....,10270
1188,r221019a_BOE,united kingdom,19/10/2022,"Governance of “Decentralised” Finance: Get up,...",wilkins,0,"These are divided into seven categories, suffr...",32759


In [7]:
# Text cleaning
df['text'] = df['text'].str.lower().str.replace('[^\w\s]', '', regex=True)

In [9]:
# VADER sentiment
sia = SentimentIntensityAnalyzer()
df[['neg', 'neu', 'pos', 'compound']] = df['text'].apply(lambda x: pd.Series(sia.polarity_scores(x)))

In [10]:
# TextBlob sentiment
df[['polarity','subjectivity']] = df['text'].apply(lambda x: pd.Series(TextBlob(x).sentiment))

In [12]:
# Load Loughran–McDonald Dictionary
lm_dict = pd.read_csv("LM_dictionary.csv")  
print("LM Columns:", lm_dict.columns)  # check columns

LM Columns: Index(['Word', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong',
       'Weak', 'Constraining'],
      dtype='object')


In [13]:
# Create a mapping: Word -> list of categories
lm_dict_map = {}
for _, row in lm_dict.iterrows():
    word = row['Word'].upper()
    categories = [col for col in lm_dict.columns[1:] if row[col] > 0]  # skip 'Word' column
    lm_dict_map[word] = categories

In [14]:
# Function to compute LM sentiment
def lm_sentiment(text, lm_dict_map):
    words = re.findall(r'\b\w+\b', text.upper())
    pos = sum(1 for w in words if 'Positive' in lm_dict_map.get(w, []))
    neg = sum(1 for w in words if 'Negative' in lm_dict_map.get(w, []))
    total = pos + neg
    return 0 if total == 0 else (pos - neg)/total

In [16]:
#import re
# Apply LM sentiment
df['lm_score'] = df['text'].apply(lambda x: lm_sentiment(x, lm_dict_map))

In [17]:
# Combined Score (simple average)
df['combined_score'] = (df['compound'] + df['polarity'] + df['lm_score']) / 3

In [24]:
# LM Label Thresholds
def lm_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    """
    Assigns a sentiment label based on LM score thresholds.
    
    Parameters:
        score: LM sentiment score ([-1,1])
        pos_thresh: threshold above which text is Positive
        neg_thresh: threshold below which text is Negative
        
    Returns:
        'Positive', 'Negative', or 'Neutral'
    """
    if score > pos_thresh:
        return "Positive"
    elif score < neg_thresh:
        return "Negative"
    else:
        return "Neutral"
    
df['lm_label'] = df['lm_score'].apply(lambda x: lm_label(x))    

# Export Selected Columns
columns_to_export = [
    'reference','date','text', 
    'neg', 'neu', 'pos', 'compound', 
    'polarity', 'subjectivity', 
    'lm_score', 
    'combined_score', 
    'lm_label'
]

In [25]:
# Export CSV
df.to_csv("financial_sentiment_labeled.csv", columns=columns_to_export, index=False, encoding='utf-8')

In [26]:
# Export Excel
df.to_excel("financial_sentiment_labeled.xlsx", columns=columns_to_export, index=False, sheet_name="Sentiment")